I want to predict all the times for one store

In [1]:
# Add project root to path so we can import from data_manipulation and model
import sys
from pathlib import Path

def _find_project_root():
    cwd = Path.cwd()
    if (cwd / "data_manipulation").is_dir():
        return cwd
    if (cwd.parent / "data_manipulation").is_dir():
        return cwd.parent
    return cwd  # fallback

project_root = _find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
import torch
from torch import nn
import matplotlib.pyplot as plt
from functools import partial

# Imports from data_manipulation and model
from data_manipulation.data_split import create_dataloader, DemandDataset
from data_manipulation.data_creation import create_data
from model.functions import pinball_loss, rmse, train, get_test_loss

In [3]:
class simple_net(nn.Module):
    def __init__(self,input_size, hidden_size=10, output_size=1):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [4]:
all_specs = [
    '6_day_lag', 
    '180_day_rolling_mean', 
    '90_day_rolling_volatility', 
    '90_day_rolling_ema', 
    'diff_30_day', 
    '7_day_rolling_mean', 
    '3_day_lag', 
    'circular_sin_cos_day_of_week', 
    '180_day_rolling_min', 
    'diff_365_day', 
    'one_hot_month', 
    '30_day_rolling_mean', 
    'circular_sin_cos_week', 
    '5_day_lag', 
    '180_day_rolling_ema']

In [8]:
# Get data for spec
train_loader, val_loader, test_loader = create_dataloader(
    batch_size=32, 
    test_batch_size=1,
    pin_memory=False,
    data_mask=[
        ("store", 1)
    ],
    specs=all_specs,
    combine_items=True
    )

         date  store  item_1  item_2  item_3  item_4  item_5  item_6  item_7  \
0  2013-01-01      1    13.0    33.0    15.0    10.0    11.0    31.0    25.0   
1  2013-01-02      1    11.0    43.0    30.0    11.0     6.0    36.0    23.0   
2  2013-01-03      1    14.0    23.0    14.0     8.0     8.0    18.0    34.0   
3  2013-01-04      1    13.0    18.0    10.0    19.0     9.0    19.0    36.0   
4  2013-01-05      1    10.0    34.0    23.0    12.0     8.0    31.0    38.0   

   item_8  ...  item_41  item_42  item_43  item_44  item_45  item_46  item_47  \
0    33.0  ...      6.0     21.0     22.0     20.0     37.0     30.0     17.0   
1    37.0  ...     15.0     24.0     27.0     15.0     40.0     30.0     15.0   
2    38.0  ...      5.0     14.0     19.0     11.0     42.0     30.0      5.0   
3    54.0  ...      9.0     22.0     29.0     22.0     49.0     37.0     13.0   
4    51.0  ...     13.0     18.0     34.0     19.0     52.0     28.0     12.0   

   item_48  item_49  item_50  
0

In [9]:
x = train_loader.dataset.x
print(x.shape)

torch.Size([1461, 1])
